In [3]:
import importlib
import snowflake_loader

importlib.reload(snowflake_loader)

print("get_connection available:", hasattr(snowflake_loader, "get_connection"))

get_connection available: True


In [4]:
conn = snowflake_loader.get_connection()

print("Snowflake connection successful")

Snowflake connection successful


In [ ]:
#configuration

import os
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_TABLE = "YIELD_FORECAST_INPUT"
RESULT_TABLE = "YIELD_FORECAST_RESULT"

MODEL_NAME = "yield_xgboost"
MODEL_VERSION = "yield_xgboost_v1"

# Paths can be changed later when P2 provides the actual model
MODEL_PATH = "./models/yield_xgboost_v1.json"

# Forecast columns
FORECAST_COLUMNS = [
    "FIELD_ID",
    "SEASON_YEAR",
    "CROP_TYPE",
    "WEEK_INDEX",
    "SEASON_DAY",
]

TARGET_COLUMN = "FORECAST_YIELD"

LOWER_BOUND_COLUMN = "FORECAST_LOWER_BOUND"
UPPER_BOUND_COLUMN = "FORECAST_UPPER_BOUND"

print("P3 configuration loaded.")

In [ ]:
# ============================================================
# LOAD FORECAST INPUT
# ============================================================

INPUT_FILE = "./data/YIELD_FORECAST_INPUT.csv"

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"YIELD_FORECAST_INPUT not found: {INPUT_FILE}"
    )

forecast_input = pd.read_csv(INPUT_FILE)

print("Shape:", forecast_input.shape)
print("\nColumns:")
print(forecast_input.columns.tolist())

forecast_input.head()

In [ ]:
# ============================================================
# INPUT VALIDATION
# ============================================================

required_columns = [
    "FIELD_ID",
    "SEASON_YEAR",
    "CROP_TYPE",
    "WEEK_INDEX",
    "SEASON_DAY",
]

missing_columns = [
    c for c in required_columns
    if c not in forecast_input.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("All required forecast input columns are present.")

In [ ]:
# ============================================================
# TEMPORAL VALIDATION
# ============================================================

temporal_check = forecast_input[
    ["FIELD_ID", "SEASON_YEAR", "WEEK_INDEX", "SEASON_DAY"]
].copy()

temporal_check["WEEK_INDEX"] = pd.to_numeric(
    temporal_check["WEEK_INDEX"],
    errors="coerce"
)

temporal_check["SEASON_DAY"] = pd.to_numeric(
    temporal_check["SEASON_DAY"],
    errors="coerce"
)

invalid_week = temporal_check["WEEK_INDEX"].isna()

invalid_day = temporal_check["SEASON_DAY"].isna()

print("Invalid WEEK_INDEX:", invalid_week.sum())
print("Invalid SEASON_DAY:", invalid_day.sum())

if invalid_week.any() or invalid_day.any():
    print("WARNING: temporal fields require cleaning.")
else:
    print("Temporal fields are valid.")

In [ ]:
# ============================================================
# MODEL FEATURE LIST
# ============================================================

FEATURE_METADATA_FILE = "./models/yield_xgboost_features.json"

if os.path.exists(FEATURE_METADATA_FILE):

    with open(FEATURE_METADATA_FILE, "r") as f:
        model_metadata = json.load(f)

    MODEL_FEATURES = model_metadata["features"]

    print("Loaded model feature list:")
    print(len(MODEL_FEATURES), "features")

else:

    # Temporary development mode
    excluded_columns = [
        "FIELD_ID",
        "SEASON_YEAR",
        "CROP_TYPE",
        "WEEK_INDEX",
        "SEASON_DAY",
        "ACTUAL_YIELD"
    ]

    MODEL_FEATURES = [
        c for c in forecast_input.columns
        if c not in excluded_columns
    ]

    print(
        "WARNING: Using temporary feature discovery."
    )

    print(
        "Temporary feature count:",
        len(MODEL_FEATURES)
    )

In [ ]:
# ============================================================
# FEATURE AVAILABILITY CHECK
# ============================================================

missing_features = [
    c for c in MODEL_FEATURES
    if c not in forecast_input.columns
]

if missing_features:
    print("Missing model features:")
    for col in missing_features:
        print(" -", col)
else:
    print("All model features are available.")

In [ ]:
# ============================================================
# MISSING FEATURE PROFILE
# ============================================================

missing_profile = pd.DataFrame({
    "FEATURE": MODEL_FEATURES,
    "MISSING_COUNT": [
        forecast_input[c].isna().sum()
        for c in MODEL_FEATURES
    ]
})

missing_profile["MISSING_PCT"] = (
    missing_profile["MISSING_COUNT"]
    / len(forecast_input)
    * 100
)

missing_profile = missing_profile.sort_values(
    "MISSING_PCT",
    ascending=False
)

display(missing_profile.head(30))

In [ ]:
# ============================================================
# FEATURE PREPARATION
# ============================================================

X = forecast_input[MODEL_FEATURES].copy()

numeric_features = X.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = [
    c for c in MODEL_FEATURES
    if c not in numeric_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

In [ ]:
# ============================================================
# NUMERIC CLEANING
# ============================================================

X_prepared = X.copy()

for col in numeric_features:
    X_prepared[col] = pd.to_numeric(
        X_prepared[col],
        errors="coerce"
    )

print("Numeric feature conversion completed.")

In [ ]:
# ============================================================
# DATA COMPLETENESS
# ============================================================

X_prepared["MISSING_FEATURE_COUNT"] = (
    X_prepared.isna().sum(axis=1)
)

X_prepared["FEATURE_COUNT"] = len(MODEL_FEATURES)

X_prepared["DATA_COMPLETENESS"] = (
    1
    - (
        X_prepared["MISSING_FEATURE_COUNT"]
        / len(MODEL_FEATURES)
    )
)

X_prepared["DATA_COMPLETENESS"] = (
    X_prepared["DATA_COMPLETENESS"]
    .clip(0, 1)
)

X_prepared[
    [
        "MISSING_FEATURE_COUNT",
        "FEATURE_COUNT",
        "DATA_COMPLETENESS"
    ]
].head()

In [ ]:
# ============================================================
# NUMERIC CLEANING
# ============================================================

X_prepared = X.copy()

for col in numeric_features:
    X_prepared[col] = pd.to_numeric(
        X_prepared[col],
        errors="coerce"
    )

print("Numeric feature conversion completed.")

In [ ]:
# ============================================================
# DATA COMPLETENESS
# ============================================================

X_prepared["MISSING_FEATURE_COUNT"] = (
    X_prepared.isna().sum(axis=1)
)

X_prepared["FEATURE_COUNT"] = len(MODEL_FEATURES)

X_prepared["DATA_COMPLETENESS"] = (
    1
    - (
        X_prepared["MISSING_FEATURE_COUNT"]
        / len(MODEL_FEATURES)
    )
)

X_prepared["DATA_COMPLETENESS"] = (
    X_prepared["DATA_COMPLETENESS"]
    .clip(0, 1)
)

X_prepared[
    [
        "MISSING_FEATURE_COUNT",
        "FEATURE_COUNT",
        "DATA_COMPLETENESS"
    ]
].head()

In [ ]:
# ============================================================
# LOAD XGBOOST MODEL
# ============================================================

try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError(
        "Install xgboost before running model inference."
    )

model = XGBRegressor()

if os.path.exists(MODEL_PATH):

    model.load_model(MODEL_PATH)

    MODEL_AVAILABLE = True

    print(
        f"Loaded model: {MODEL_NAME} "
        f"{MODEL_VERSION}"
    )

else:

    MODEL_AVAILABLE = False

    print(
        "Model file not available yet."
    )

    print(
        "This is expected until Person 2 "
        "provides the approved XGBoost model."
    )

In [ ]:
# ============================================================
# DEVELOPMENT MOCK MODEL
# ============================================================

class MockYieldModel:

    def predict(self, X):
        """
        Development-only placeholder.

        Produces a constant prediction so that
        the complete P3 pipeline can be tested.
        """

        return np.full(
            len(X),
            3.5,
            dtype=float
        )


if not MODEL_AVAILABLE:

    model = MockYieldModel()

    MODEL_NAME = "mock_yield_model"
    MODEL_VERSION = "mock_v0"

    print(
        "WARNING: Mock model active."
    )

    print(
        "Do NOT use these predictions for production."
    )

In [ ]:
# ============================================================
# GENERATE YIELD FORECAST
# ============================================================

prediction_input = forecast_input[MODEL_FEATURES].copy()

# Convert numeric fields
for col in numeric_features:
    prediction_input[col] = pd.to_numeric(
        prediction_input[col],
        errors="coerce"
    )

forecast_prediction = model.predict(
    prediction_input
)

forecast_prediction = np.asarray(
    forecast_prediction,
    dtype=float
)

print(
    "Predictions generated:",
    len(forecast_prediction)
)

print(
    "Prediction range:",
    forecast_prediction.min(),
    "to",
    forecast_prediction.max()
)

In [ ]:
# ============================================================
# TEMPORARY UNCERTAINTY METHOD
# ============================================================

DEFAULT_INTERVAL_PCT = 0.10

forecast_lower = (
    forecast_prediction
    * (1 - DEFAULT_INTERVAL_PCT)
)

forecast_upper = (
    forecast_prediction
    * (1 + DEFAULT_INTERVAL_PCT)
)

forecast_lower = np.maximum(
    forecast_lower,
    0
)

forecast_upper = np.maximum(
    forecast_upper,
    forecast_lower
)

print("Temporary forecast bounds created.")

In [ ]:
# ============================================================
# FORECAST CONFIDENCE
# ============================================================

forecast_confidence = (
    X_prepared["DATA_COMPLETENESS"]
    .astype(float)
    .clip(0, 1)
)

print(
    forecast_confidence.describe()
)

In [ ]:
# ============================================================
# BUILD FINAL RESULT
# ============================================================

result = forecast_input[
    FORECAST_COLUMNS
].copy()

result["FORECAST_YIELD"] = forecast_prediction

result["FORECAST_LOWER_BOUND"] = forecast_lower

result["FORECAST_UPPER_BOUND"] = forecast_upper

result["FORECAST_CONFIDENCE"] = (
    forecast_confidence.values
)

result["MODEL_NAME"] = MODEL_NAME

result["MODEL_VERSION"] = MODEL_VERSION

result["FEATURE_COUNT"] = (
    X_prepared["FEATURE_COUNT"].values
)

result["MISSING_FEATURE_COUNT"] = (
    X_prepared["MISSING_FEATURE_COUNT"].values
)

result["DATA_COMPLETENESS"] = (
    X_prepared["DATA_COMPLETENESS"].values
)

result["FORECAST_GENERATED_AT"] = (
    pd.Timestamp.utcnow()
)

print(
    "YIELD_FORECAST_RESULT shape:",
    result.shape
)

display(result.head())

In [ ]:
# ============================================================
# RESULT VALIDATION
# ============================================================

validation_results = {}

validation_results["NULL_FORECAST"] = (
    result["FORECAST_YIELD"].isna().sum()
)

validation_results["NEGATIVE_FORECAST"] = (
    (result["FORECAST_YIELD"] < 0).sum()
)

validation_results["INVALID_LOWER_BOUND"] = (
    (
        result["FORECAST_LOWER_BOUND"]
        > result["FORECAST_YIELD"]
    ).sum()
)

validation_results["INVALID_UPPER_BOUND"] = (
    (
        result["FORECAST_UPPER_BOUND"]
        < result["FORECAST_YIELD"]
    ).sum()
)

validation_results["INVALID_CONFIDENCE"] = (
    (
        (result["FORECAST_CONFIDENCE"] < 0)
        |
        (result["FORECAST_CONFIDENCE"] > 1)
    ).sum()
)

validation_results["DUPLICATE_GRAIN"] = (
    result.duplicated(
        subset=[
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX"
        ]
    ).sum()
)

validation_results

In [ ]:
# ============================================================
# VALIDATION GATE
# ============================================================

validation_failed = any(
    value > 0
    for value in validation_results.values()
)

if validation_failed:

    print("❌ FORECAST VALIDATION FAILED")

    for check, count in validation_results.items():

        if count > 0:
            print(
                f"{check}: {count}"
            )

else:

    print(
        "✅ FORECAST VALIDATION PASSED"
    )

In [ ]:
# ============================================================
# FORECAST HISTORY
# ============================================================

HISTORY_FILE = "./data/YIELD_FORECAST_HISTORY.csv"

os.makedirs(
    os.path.dirname(HISTORY_FILE),
    exist_ok=True
)

if os.path.exists(HISTORY_FILE):

    history = pd.read_csv(
        HISTORY_FILE
    )

else:

    history = pd.DataFrame()

print(
    "Existing history rows:",
    len(history)
)

In [ ]:
# ============================================================
# UPSERT-STYLE FORECAST HISTORY
# ============================================================

history_keys = [
    "FIELD_ID",
    "SEASON_YEAR",
    "WEEK_INDEX",
    "MODEL_VERSION"
]

if len(history) > 0:

    history = history[
        ~history.set_index(history_keys).index.isin(
            result.set_index(history_keys).index
        )
    ]

history = pd.concat(
    [
        history,
        result
    ],
    ignore_index=True
)

history = (
    history
    .drop_duplicates(
        subset=history_keys,
        keep="last"
    )
)

history.to_csv(
    HISTORY_FILE,
    index=False
)

print(
    "Forecast history rows:",
    len(history)
)

In [ ]:
# ============================================================
# FORECAST EVOLUTION
# ============================================================

history_sorted = history.sort_values(
    [
        "FIELD_ID",
        "SEASON_YEAR",
        "WEEK_INDEX"
    ]
)

display(
    history_sorted[
        [
            "FIELD_ID",
            "SEASON_YEAR",
            "WEEK_INDEX",
            "FORECAST_YIELD",
            "FORECAST_LOWER_BOUND",
            "FORECAST_UPPER_BOUND",
            "FORECAST_CONFIDENCE",
            "MODEL_VERSION"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# REUSABLE INFERENCE FUNCTION
# ============================================================

def generate_yield_forecast(
    input_df,
    model,
    model_name,
    model_version,
    model_features,
    interval_pct=0.10
):
    """
    Generate yield forecast for the supplied
    current-season input data.
    """

    required = [
        "FIELD_ID",
        "SEASON_YEAR",
        "CROP_TYPE",
        "WEEK_INDEX",
        "SEASON_DAY"
    ]

    missing = [
        c for c in required
        if c not in input_df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    missing_features = [
        c for c in model_features
        if c not in input_df.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing model features: {missing_features}"
        )

    X = input_df[
        model_features
    ].copy()

    numeric_cols = X.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_cols:

        X[col] = pd.to_numeric(
            X[col],
            errors="coerce"
        )

    missing_count = X.isna().sum(axis=1)

    completeness = (
        1
        - missing_count / len(model_features)
    ).clip(0, 1)

    prediction = np.asarray(
        model.predict(X),
        dtype=float
    )

    lower = np.maximum(
        prediction * (1 - interval_pct),
        0
    )

    upper = np.maximum(
        prediction * (1 + interval_pct),
        lower
    )

    output = input_df[
        required
    ].copy()

    output["FORECAST_YIELD"] = prediction

    output["FORECAST_LOWER_BOUND"] = lower

    output["FORECAST_UPPER_BOUND"] = upper

    output["FORECAST_CONFIDENCE"] = completeness

    output["MODEL_NAME"] = model_name

    output["MODEL_VERSION"] = model_version

    output["FEATURE_COUNT"] = len(
        model_features
    )

    output["MISSING_FEATURE_COUNT"] = (
        missing_count.values
    )

    output["DATA_COMPLETENESS"] = (
        completeness.values
    )

    output["FORECAST_GENERATED_AT"] = (
        pd.Timestamp.utcnow()
    )

    return output

In [ ]:
# ============================================================
# TEST INFERENCE FUNCTION
# ============================================================

test_result = generate_yield_forecast(
    input_df=forecast_input,
    model=model,
    model_name=MODEL_NAME,
    model_version=MODEL_VERSION,
    model_features=MODEL_FEATURES,
    interval_pct=0.10
)

print(
    "Generated:",
    len(test_result),
    "forecast records"
)

display(test_result.head())